In [ ]:
import pandas            as pd
import numpy             as np
import seaborn           as sns
import matplotlib.pyplot as plt
import seaborn           as sns
import tensorflow        as tf
import math
import heapq

from tensorflow.keras.models       import Sequential
from tensorflow.keras              import layers
from tensorflow.keras.backend      import sqrt
from tensorflow.keras.losses       import MeanSquaredLogarithmicError
from tensorflow.keras.optimizers   import RMSprop
from tensorflow.keras              import regularizers
from sklearn.preprocessing         import RobustScaler,MinMaxScaler,KBinsDiscretizer, LabelEncoder
from sklearn.pipeline              import Pipeline
from sklearn.model_selection       import train_test_split
from sklearn                       import metrics
import tensorflow.keras.backend     as K

import  os 

In [ ]:
def root_mean_squared_log_error(y_true, y_pred):
    return K.sqrt(K.mean(K.square(K.log(1+y_pred) - K.log(1+y_true))))

In [ ]:
def generator(data,lookback,delay,min_index,max_index,
             target,shuffle=False,batch_size=128,step=6):
    if max_index is None:
        max_index = len(data) - delay - 1
    if (min_index < lookback):
        i = min_index + lookback
    else:
        i = min_index
    while 1:
        if shuffle:
            rows = np.random.randint(min_index + lookback, max_index, size=batch_size)
        else:
            if i + batch_size >= max_index:
                if (min_index < lookback):
                    i = min_index + lookback
                else:
                    i = min_index
            rows = np.arange(i, min(i + batch_size, max_index))
            i += len(rows)
        samples = np.zeros((len(rows),
                           lookback // step,
                           data.shape[-1]-len(target)))
        targets = np.zeros((len(rows),len(target),))
        cols = [col for col in data.columns if col not in target]
        for j, row in enumerate(rows):
            indices = range(rows[j] - lookback, rows[j], step)
            samples[j] = data[cols].iloc[indices,:]
            targets[j] = data[target].iloc[rows[j] + delay,:]
        yield samples, targets   

In [ ]:
holiday = pd.read_csv("../input/store-sales-time-series-forecasting/holidays_events.csv")
oil  = pd.read_csv("../input/store-sales-time-series-forecasting/oil.csv")
stores = pd.read_csv("../input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("../input/store-sales-time-series-forecasting/transactions.csv")
train = pd.read_csv("../input/store-sales-time-series-forecasting/train.csv")
test = pd.read_csv("../input/store-sales-time-series-forecasting/test.csv")
sample_submission = pd.read_csv("../input/store-sales-time-series-forecasting/sample_submission.csv")

In [ ]:
#investigating the files
files = [holiday,oil,stores,transactions,train,test,sample_submission]
titles = ["Holiday","Oil","Stores","Transactions","Train","Test","Sample_submission"]

for file, title in zip(files,titles):
    print("##################")
    print(title)
    print(file.columns)
    #print(file)

In [ ]:
for file, title in zip(files,titles):
    print("##################")
    print(title)
    object_cols = []
    number_cols = []
    for col in file.columns:
        if (file[col].dtype == 'object'):
            object_cols.append(col)
        else:
            number_cols.append(col)
    print("{} Object Columns".format(title))
    print(object_cols)
    print("{} Numerical Columns".format(title))
    print(number_cols)
    #print(file[number_cols])
    for col in object_cols:
        print("Unique values in column {} of {}:".format(col,file))
        print(file[col].value_counts())
        #print("Number of unique values".format(len(file[col].unique())))
        print("******************")

In [ ]:
## combine datasets
train = train.merge(holiday, on = 'date', how='left')
train = train.merge(oil, on = 'date', how='left')
train = train.merge(stores, on = 'store_nbr', how='left')
train = train.merge(transactions, on = ['date', 'store_nbr'], how='left')
print(train.columns)
train = train.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

test = test.merge(holiday, on = 'date', how='left')
test = test.merge(oil, on = 'date', how='left')
test = test.merge(stores, on = 'store_nbr', how='left')
test = test.merge(transactions, on = ['date', 'store_nbr'], how='left')
test = test.rename(columns = {"type_x" : "holiday_type", "type_y" : "store_type"})

In [ ]:
train['date_parsed'] = pd.to_datetime(train['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(train['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
test['date_parsed'] = pd.to_datetime(test['date'],format="%Y-%m-%d")
not_datetime = pd.to_datetime(test['date'], errors='coerce',format="%Y-%m-%d").isna()
print("Rows in training data with wrong date and time format:")
print(not_datetime[not_datetime == True])

In [ ]:
train['quarter']    = train['date_parsed'].dt.quarter
train['month']      = train['date_parsed'].dt.month
train['year']       = train['date_parsed'].dt.year
train['dayofyear']  = train['date_parsed'].dt.dayofyear
train['weekofyear'] = train['date_parsed'].dt.isocalendar().week
#train['date']       = train['date_parsed'].values.astype('float64')

In [ ]:
test['quarter']    = test['date_parsed'].dt.quarter
test['month']      = test['date_parsed'].dt.month
test['year']       = test['date_parsed'].dt.year
test['dayofyear']  = test['date_parsed'].dt.dayofyear
test['weekofyear'] = test['date_parsed'].dt.isocalendar().week
#test['date']       = test['date_parsed'].values.astype('float64')

In [ ]:
train.drop(['date_parsed'],axis=1,inplace=True)
test.drop(['date_parsed'],axis=1,inplace=True)

In [ ]:
print("Train columns:")
print(train.columns)

In [ ]:
train[['store_nbr', 'cluster']] = train[['store_nbr', 'cluster']].astype(str)
test[['store_nbr', 'cluster']] = test[['store_nbr', 'cluster']].astype(str)

In [ ]:
## Item Categories object and number columns
object_cols = []
number_cols = []
for col in train.columns:
    if (train[col].dtype == 'object'):
        object_cols.append(col)
    else:
        number_cols.append(col)
print("Training Object Columns")
print(object_cols)
print("Training Numerical Columns")
print(number_cols)

In [ ]:
agg = train.groupby('date').agg({"sales" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="sales",title = "Average Sales by Date")

agg = train.groupby('date').agg({"transactions" : "mean"}).reset_index()
fig = agg.plot(kind="line", x="date", y="transactions",title = "Average Transactions by Date")

In [ ]:
agg = train.groupby(['date', 'store_type']).agg({"sales" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

agg = train.groupby(['date', 'store_type']).agg({"transactions" : "mean"}).reset_index()
agg.pivot(columns='store_type').interpolate().plot()

In [ ]:
train.drop(['date'],axis=1,inplace=True)
test.drop(['date'], axis=1,inplace=True)

In [ ]:
y = train['sales']
X = train
X.drop(['sales'],axis=1,inplace=True) 

In [ ]:
train[['holiday_type','locale','locale_name','description']] = train[['holiday_type','locale','locale_name','description']].fillna("Work Day")
test[['holiday_type','locale','locale_name','description']] = test[['holiday_type','locale','locale_name','description']].fillna("Work Day")
train['transferred'] = train['transferred'].fillna(False)
test['transferred'] = test['transferred'].fillna(False)
train['dcoilwtico'] = train['dcoilwtico'].fillna((train['dcoilwtico'].mean()), inplace=False)
test['dcoilwtico'] = test['dcoilwtico'].fillna((test['dcoilwtico'].mean()), inplace=False)

In [ ]:
missing_data_count = train.isnull().sum()
print("Missing Data in Training Set:")
print(missing_data_count)
sns.heatmap(train.isnull())
plt.show()

In [ ]:
missing_data_count = test.isnull().sum()
print("Missing Data in Test Set:")
print(missing_data_count)
sns.heatmap(test.isnull())
plt.show()

In [ ]:
XandTest = pd.concat([X,test])
s = (X.dtypes == 'object')
object_cols = list(s[s].index)
print("Object Columns")
print(object_cols)
label_encoder = LabelEncoder()
for col in object_cols:
    if (col == 'date_parsed'):
        continue
    XandTest[col] = label_encoder.fit_transform(XandTest[col])
    X[col] = label_encoder.transform(X[col])
    test[col]  = label_encoder.transform(test[col])

In [ ]:
columns  = X.columns
pipe     = Pipeline([("scaler_1", RobustScaler()),("scaler_2", MinMaxScaler())])
XandTest = pd.concat([X,test])
XandTest = pipe.fit(XandTest)
X        = pipe.transform(X)
test     = pipe.transform(test)
X        = pd.DataFrame(X,   columns=columns)
test     = pd.DataFrame(test,columns=columns)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25)

In [ ]:
model = Sequential()
#model.add(layers.LSTM(64,activation='selu',input_shape=(X.shape[1:],),return_sequences=False)) #, kernel_regularizer=regularizers.l2(0.001))
model.add(layers.Dense(1024,activation='selu',input_shape=(X.shape[1],))) #, kernel_regularizer=regularizers.l2(0.001))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(1024,activation='selu')) #, kernel_regularizer=regularizers.l2(0.001))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(1))
print(model.summary())

In [ ]:
#lrs = [1e-4,5e-4,1e-3,2e-3,3e-3,5e-3,8e-3,9e-3,1e-2,5e-2,1e-1,5e-1]
#for lr in lrs:
model.compile(optimizer=RMSprop(),loss=root_mean_squared_log_error,metrics=['mean_squared_error'])
history = model.fit(X_train,y_train,epochs=50,batch_size=4196,validation_data=(X_valid,y_valid))
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
from math import sqrt
loss = [sqrt(i) for i in loss]
val_loss = [sqrt(i) for i in val_loss]
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss for learning rate of {}'.format(lr))
#plt.ylim(6,10)
plt.legend()
plt.show()